dataset generation

In [1]:
import requests
import pandas as pd
import numpy as np
import os

os.makedirs("data", exist_ok=True)

url = "https://power.larc.nasa.gov/api/temporal/daily/point"

params = {
    "parameters": "ALLSKY_SFC_SW_DWN,T2M,WS2M",
    "community": "RE",
    "latitude": 11.92237,
    "longitude": 79.60673,
    "start": 20250101,
    "end": 20251231,
    "format": "JSON"
}

response = requests.get(url, params=params)
data = response.json()

irradiance = data["properties"]["parameter"]["ALLSKY_SFC_SW_DWN"]
temp = data["properties"]["parameter"]["T2M"]
wind = data["properties"]["parameter"]["WS2M"]

rows = []
prev_efficiency = 0

# 🔥 global max for scaling
max_irrad = max(irradiance.values())

for date in irradiance.keys():

    daily_irrad = irradiance[date]
    temperature = temp[date]
    wind_speed = wind[date]

    for hour in range(24):

        # ❌ skip night (important)
        if not (6 <= hour <= 18):
            continue

        # solar curve
        factor = np.sin((hour - 6) / 12 * np.pi)

        solar_output = daily_irrad * factor

        efficiency = (solar_output / max_irrad) * 100
        efficiency = max(0, min(efficiency, 100))

        rows.append([
            date,
            hour,
            temperature,
            wind_speed,
            daily_irrad,
            prev_efficiency,
            round(efficiency, 2)
        ])

        prev_efficiency = efficiency

df = pd.DataFrame(rows, columns=[
    "date",
    "hour",
    "temperature",
    "wind_speed",
    "irradiance",
    "historical_efficiency",
    "efficiency"
])

df.to_csv("data/solar.csv", index=False)

print("🔥 Dataset created with proper efficiency!")

🔥 Dataset created with proper efficiency!


Training

In [8]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
import numpy as np
import joblib

# -----------------------------
# LOAD DATA
# -----------------------------
df = pd.read_csv("data/solar.csv")

# -----------------------------
# FEATURE ENGINEERING
# -----------------------------
df["date"] = pd.to_datetime(df["date"])
df["month"] = df["date"].dt.month

features = [
    "month",
    "hour",
    "temperature",
    "wind_speed",
    "irradiance",
    "historical_efficiency"
]

X = df[features]
y = df["efficiency"]

# -----------------------------
# TRAIN / TEST SPLIT
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# -----------------------------
# TRAIN MODEL
# -----------------------------
model = RandomForestRegressor(
    n_estimators=200,
    max_depth=12,
    random_state=42
)

model.fit(X_train, y_train)

# -----------------------------
# EVALUATION
# -----------------------------
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
accuracy = (1 - mae/100) * 100

print("\n📊 MODEL PERFORMANCE (REAL)")
print("MAE :", round(mae, 2))
print("RMSE:", round(rmse, 2))
print("R2  :", round(r2, 3))
print("acc:", accuracy)

# -----------------------------
# SAVE MODEL
# -----------------------------
joblib.dump(model, "models/solar.pkl")

print("\n✅ Model trained  with validation!")


📊 MODEL PERFORMANCE (REAL)
MAE : 0.12
RMSE: 0.25
R2  : 1.0
acc: 99.88210559456188

✅ Model trained  with validation!


Predict

In [9]:
import joblib
import pandas as pd

# -----------------------------
# LOAD MODEL
# -----------------------------
model = joblib.load("models/solar.pkl")

# -----------------------------
# PANEL CONFIG
# -----------------------------
PANEL_CAPACITY_KW = 1.0

# -----------------------------
# INPUT
# -----------------------------
input_data ={
            "month": 5,
            "hour": 12,
            "temperature": 35,
            "wind_speed": 3,
            "irradiance": 2.01,
            "historical_efficiency": 70
        }

df = pd.DataFrame([input_data])

# -----------------------------
# PREDICT
# -----------------------------
efficiency = model.predict(df)[0]

# clamp
efficiency = max(0, min(efficiency, 100))

power_output = (efficiency / 100) * PANEL_CAPACITY_KW

# -----------------------------
# OUTPUT
# -----------------------------
print(f"\n⚡ Efficiency: {round(efficiency,2)} %")
print(f"☀️ Solar Output: {round(power_output,2)} kW")


⚡ Efficiency: 57.4 %
☀️ Solar Output: 0.57 kW


Scenario Test

In [1]:
import joblib
import pandas as pd

# -----------------------------
# LOAD MODEL
# -----------------------------
model = joblib.load("models/solar.pkl")

# -----------------------------
# PANEL CONFIG
# -----------------------------
PANEL_CAPACITY_KW = 1.0

# -----------------------------
# TEST SCENARIOS
# -----------------------------
scenarios = [
    {
        "name": "☀️ Sunny Noon",
        "input": {
            "month": 5,
            "hour": 12,
            "temperature": 35,
            "wind_speed": 3,
            "irradiance": 900,
            "historical_efficiency": 70
        }
    },
    {
        "name": "🌤️ Morning",
        "input": {
            "month": 5,
            "hour": 8,
            "temperature": 28,
            "wind_speed": 2,
            "irradiance": 500,
            "historical_efficiency": 40
        }
    },
    {
        "name": "🌥️ Cloudy Afternoon",
        "input": {
            "month": 5,
            "hour": 14,
            "temperature": 30,
            "wind_speed": 4,
            "irradiance": 300,
            "historical_efficiency": 35
        }
    },
    {
        "name": "🌅 Evening",
        "input": {
            "month": 5,
            "hour": 17,
            "temperature": 29,
            "wind_speed": 3,
            "irradiance": 200,
            "historical_efficiency": 20
        }
    },
    {
        "name": "🌙 Night",
        "input": {
            "month": 5,
            "hour": 22,
            "temperature": 27,
            "wind_speed": 2,
            "irradiance": 0,
            "historical_efficiency": 5
        }
    },
    {
        "name": "❄️ Winter Noon",
        "input": {
            "month": 12,
            "hour": 12,
            "temperature": 25,
            "wind_speed": 3,
            "irradiance": 600,
            "historical_efficiency": 50
        }
    }
]

# -----------------------------
# RUN TESTS
# -----------------------------
print("\n🔍 SOLAR MODEL TEST SCENARIOS\n")

for scenario in scenarios:

    df = pd.DataFrame([scenario["input"]])

    efficiency = model.predict(df)[0]
    efficiency = max(0, min(efficiency, 100))

    power_output = (efficiency / 100) * PANEL_CAPACITY_KW

    print(f"--- {scenario['name']} ---")
    print(f"Efficiency: {round(efficiency,2)} %")
    print(f"Power     : {round(power_output,2)} kW\n")


🔍 SOLAR MODEL TEST SCENARIOS

--- ☀️ Sunny Noon ---
Efficiency: 85.99 %
Power     : 0.86 kW

--- 🌤️ Morning ---
Efficiency: 59.13 %
Power     : 0.59 kW

--- 🌥️ Cloudy Afternoon ---
Efficiency: 33.24 %
Power     : 0.33 kW

--- 🌅 Evening ---
Efficiency: 12.06 %
Power     : 0.12 kW

--- 🌙 Night ---
Efficiency: 0 %
Power     : 0.0 kW

--- ❄️ Winter Noon ---
Efficiency: 69.94 %
Power     : 0.7 kW



In [7]:
import pandas as pd

df = pd.read_csv("data/anomaly.csv")
print(df.describe())

              hour      voltage      current  consumption      anomaly
count  2000.000000  2000.000000  2000.000000  2000.000000  2000.000000
mean     11.468000   224.735001     2.975546     0.669025     0.048500
std       6.930006     8.779324     1.942815     0.439131     0.214874
min       0.000000   210.007502     0.500205     0.110000     0.000000
25%       5.000000   216.796702     1.628515     0.360000     0.000000
50%      11.000000   224.524535     2.737378     0.610000     0.000000
75%      17.000000   232.252208     4.005163     0.900000     0.000000
max      23.000000   239.999356    17.342878     4.040000     1.000000


In [1]:
print(1)

1
